In [0]:
import random
from datetime import datetime, timedelta

random.seed(42)

NOMINAL_PRESSURE = 1200.0
NOMINAL_FLOW = 1500.0

def make_reading(well_id, ts, drop=False):
    """Return one raw reading as a dict of strings."""
    pressure = NOMINAL_PRESSURE + random.uniform(-10, 10)
    flow = NOMINAL_FLOW + random.uniform(-20, 20)
    if drop:
        pressure -= 450
        flow -= 850

    r = random.random()
    if r < 0.03:
        pressure_raw = ""
    elif r < 0.06:
        pressure_raw = "ERR_VAL_99"
    elif r < 0.09:
        pressure_raw = "-999.0"
    else:
        pressure_raw = str(round(pressure, 2))

    return {
        "well_id": well_id,
        "sensor_timestamp": ts.strftime("%Y-%m-%d %H:%M:%S"),
        "wellhead_pressure_raw": pressure_raw,
        "flow_bpd_raw": str(round(flow, 2)),
    }

In [0]:
for i in range(5):
    print(make_reading("WELL_01", datetime(2026, 1, 1) + timedelta(minutes=10 * i)))

In [0]:
import json, os

VOLUME_PATH = "/Volumes/workspace/oil_gas_bronze/landing/telemetry"
os.makedirs(VOLUME_PATH, exist_ok=True)

wells = [f"WELL_{i:02d}" for i in range(1, 6)]
start = datetime(2026, 1, 1)
step = timedelta(minutes=10)
drops = {
    "WELL_03": (start + timedelta(days=8), start + timedelta(days=9)),
    "WELL_05": (start + timedelta(days=11), start + timedelta(days=12)),
}

def in_drop(well_id, ts):
    window = drops.get(well_id)
    return window is not None and window[0] <= ts < window[1]

random.seed(42)
for well_id in wells:
    for d in range(14):
        day_start = start + timedelta(days=d)
        ts = day_start
        path = f"{VOLUME_PATH}/{well_id}_{day_start:%Y%m%d}.json"
        with open(path, "w") as f:
            while ts < day_start + timedelta(days=1):
                f.write(json.dumps(make_reading(well_id, ts, drop=in_drop(well_id, ts))) + "\n")
                ts += step

print("files:", len(os.listdir(VOLUME_PATH)))

In [0]:
print(spark.read.text(VOLUME_PATH).count())
